# Prediction Bias
Let's look at the prediction bias for Payer Model Test

## Imports

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


In [ ]:
import numpy as np
import sqlparse
import pandas as pd
from google.cloud import bigquery

%load_ext google.colab.data_table
import plotly.express as px

In [ ]:
# %%bigquery --project unity-ads-ds-prd
# SELECT * FROM unity-ai-data-prd.ads_events_raw.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS
# where table_name='ads_events_operativeecpm_v1'

In [ ]:
# %%bigquery --project unity-ads-ds-prd
# SELECT * FROM unity-ads-bi-prd.rawevents.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS
# where table_name='operativeecpm_installs_outcomes_contextual'

In [ ]:
# %%bigquery --project unity-ads-ds-prd
# SELECT * FROM unity-ai-data-prd.ads_events_raw.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS
# where table_name='ads_events_operativeecpm_v0'

In [ ]:
# %%bigquery --project unity-ads-ds-prd
# SELECT * FROM unity-ads-bi-prd.rawevents.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS
# where table_name='operativeecpm_installs_outcomes_contextual'

In [ ]:
# %%bigquery --project unity-ads-ds-prd
# SELECT * FROM unity-ai-data-prd.mz_dcpi_raw.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS
# where table_name='mz_dcpi_valuation_v1'
# # operativeecpm_installs_outcomes_contextual

In [ ]:
# %%bigquery --project unity-ads-ds-prd
# SELECT * FROM unity-ai-data-prd.mz_dcpi_raw.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS
# where table_name='mz_dcpi_valuation_v1'

#v11_payer - Bias exploration Query

In [ ]:
%%bigquery payer_df_all --project unity-ads-bi-prd
WITH PREDS as (
  SELECT
    submit_date,
    -- Updated version logic using CASE and LIKE
    CASE
      WHEN body.app_event_model_version LIKE '%v11-payer%' THEN 'v11-payer'
      WHEN body.app_event_model_version LIKE '%ctx1i-1a%' THEN 'ctx1i'
      WHEN body.app_event_model_version LIKE '%bhv1n-1a%' THEN 'bhv1n'
      ELSE null
    END as app_event_model_version,
    body.auction_id,
    body.app_event_p as pred,
    body.max_cst as target_cpe,
    body.cst as cost,
  FROM `unity-ai-data-prd.mz_dcpi_raw.mz_dcpi_prediction_v1`
  WHERE (submit_date >= "2026-03-29" and submit_date <= "2026-04-12")
  AND body.app_event_p > 0
  AND body.app_event_type = "purchase"
),

OUTCOMES AS (
  SELECT
    auctionId,
    campaignSpend,
    cum_deposit_capped_count_d0,
    cum_deposit_capped_count_d1,
    cum_deposit_capped_count_d2,
    cum_deposit_capped_count_d3,
    cum_deposit_capped_count_d4,
    cum_deposit_capped_count_d5,
    cum_deposit_capped_count_d6,
    cum_deposit_capped_count_d7,

    (CASE WHEN cum_deposit_capped_count_d0 > 0 THEN 1 ELSE 0 END) AS payer_label_d0,
    (CASE WHEN cum_deposit_capped_count_d1 > 0 THEN 1 ELSE 0 END) AS payer_label_d1,
    (CASE WHEN cum_deposit_capped_count_d2 > 0 THEN 1 ELSE 0 END) AS payer_label_d2,
    (CASE WHEN cum_deposit_capped_count_d3 > 0 THEN 1 ELSE 0 END) AS payer_label_d3,
    (CASE WHEN cum_deposit_capped_count_d4 > 0 THEN 1 ELSE 0 END) AS payer_label_d4,
    (CASE WHEN cum_deposit_capped_count_d5 > 0 THEN 1 ELSE 0 END) AS payer_label_d5,
    (CASE WHEN cum_deposit_capped_count_d6 > 0 THEN 1 ELSE 0 END) AS payer_label_d6,
    (CASE WHEN cum_deposit_capped_count_d7 > 0 THEN 1 ELSE 0 END) AS payer_label_d7,

    cum_deposit_capped_sum_d0,
    cum_deposit_capped_sum_d1,
    cum_deposit_capped_sum_d2,
    cum_deposit_capped_sum_d3,
    cum_deposit_capped_sum_d4,
    cum_deposit_capped_sum_d5,
    cum_deposit_capped_sum_d6,
    cum_deposit_capped_sum_d7,
    FROM `unity-data-ads-core-prd.ads_secondary_conversion.operativeecpm_installs_outcomes_contextual`
    WHERE (adRequestTimestamp >= TIMESTAMP("2026-03-29") and adRequestTimestamp <= TIMESTAMP("2026-04-12"))
)

SELECT
  submit_date,
  app_event_model_version,
  COUNT(*) as installs,
  SUM(pred) as sum_pred,
  AVG(pred) as avg_pred,

  SUM(target_cpe) as sum_tcpe,
  AVG(target_cpe) as avg_tcpe,
  SUM(cost) as sum_cost,
  AVG(cost) as avg_cost,

  SUM(campaignSpend) as sum_campaign_spend,

  SUM(payer_label_d0) as sum_payer_label_d0,
  SUM(payer_label_d1) as sum_payer_label_d1,
  SUM(payer_label_d2) as sum_payer_label_d2,
  SUM(payer_label_d3) as sum_payer_label_d3,
  SUM(payer_label_d4) as sum_payer_label_d4,
  SUM(payer_label_d5) as sum_payer_label_d5,
  SUM(payer_label_d6) as sum_payer_label_d6,
  SUM(payer_label_d7) as sum_payer_label_d7,

  SUM(cum_deposit_capped_count_d0) as sum_capped_count_d0,
  SUM(cum_deposit_capped_count_d1) as sum_capped_count_d1,
  SUM(cum_deposit_capped_count_d2) as sum_capped_count_d2,
  SUM(cum_deposit_capped_count_d3) as sum_capped_count_d3,
  SUM(cum_deposit_capped_count_d4) as sum_capped_count_d4,
  SUM(cum_deposit_capped_count_d5) as sum_capped_count_d5,
  SUM(cum_deposit_capped_count_d6) as sum_capped_count_d6,
  SUM(cum_deposit_capped_count_d7) as sum_capped_count_d7,

  SUM(cum_deposit_capped_sum_d0) as sum_capped_deposit_d0,
  SUM(cum_deposit_capped_sum_d1) as sum_capped_deposit_d1,
  SUM(cum_deposit_capped_sum_d2) as sum_capped_deposit_d2,
  SUM(cum_deposit_capped_sum_d3) as sum_capped_deposit_d3,
  SUM(cum_deposit_capped_sum_d4) as sum_capped_deposit_d4,
  SUM(cum_deposit_capped_sum_d5) as sum_capped_deposit_d5,
  SUM(cum_deposit_capped_sum_d6) as sum_capped_deposit_d6,
  SUM(cum_deposit_capped_sum_d7) as sum_capped_deposit_d7,
FROM PREDS
-- Changed to INNER JOIN or ensure PREDS is LEFT to keep the mapped versions consistent
RIGHT JOIN OUTCOMES ON OUTCOMES.auctionId = PREDS.auction_id
WHERE pred > 0
GROUP BY 1, 2
ORDER BY 1, 2

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
payer_df_all

,submit_date,app_event_model_version,installs,sum_pred,avg_pred,sum_tcpe,avg_tcpe,sum_cost,avg_cost,sum_campaign_spend,...,sum_capped_count_d6,sum_capped_count_d7,sum_capped_deposit_d0,sum_capped_deposit_d1,sum_capped_deposit_d2,sum_capped_deposit_d3,sum_capped_deposit_d4,sum_capped_deposit_d5,sum_capped_deposit_d6,sum_capped_deposit_d7
0,2026-03-29,bhv1n,869,40.027658,0.046062,53763940000,6.186875e+07,2258441514,2.598897e+06,2258441514,...,33,35,61.807570,1511.393094,2946.559830,2950.298391,5465.117863,7261.106486,7320.358486,7336.248486
1,2026-03-29,ctx1i,899,59.795531,0.066513,43412650000,4.828993e+07,2503638045,2.784914e+06,2503638045,...,48,52,89.729859,90.944859,90.944859,90.944859,90.944859,90.944859,403.744859,517.274859
2,2026-03-29,v11-payer,1275,89.721261,0.070370,72520750000,5.687902e+07,5176174576,4.059745e+06,5176174576,...,64,67,125.362501,136.101501,136.101501,136.101501,68.973891,71.457891,345.267891,371.952132
3,2026-03-30,bhv1n,967,44.974848,0.046510,65889550000,6.813811e+07,2843675566,2.940719e+06,2843675566,...,26,26,669.206176,3315.300737,3336.290737,3336.290737,3336.290737,3336.290737,4077.258541,4077.258541
4,2026-03-30,ctx1i,751,48.297409,0.064311,34872880000,4.643526e+07,1836228994,2.445045e+06,1836228994,...,24,24,44.000000,44.000000,44.000000,44.000000,44.000000,60.000000,106.530000,106.530000
5,2026-03-30,v11-payer,976,68.710391,0.070400,54316530000,5.565218e+07,3826569949,3.920666e+06,3826569949,...,53,56,60.443832,60.443832,60.443832,61.678832,76.419921,85.836921,269.326921,390.356921
6,2026-03-31,bhv1n,778,38.430931,0.049397,56160540000,7.218578e+07,2487350744,3.197109e+06,2487350744,...,31,31,92.379082,98.356082,128.326082,128.326082,128.326082,128.326082,141.316082,141.316082
7,2026-03-31,ctx1i,685,47.149240,0.068831,32594880000,4.758377e+07,1923600468,2.808176e+06,1923600468,...,21,22,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000,223.320000,231.710000
8,2026-03-31,v11-payer,801,58.506729,0.073042,44542960000,5.560919e+07,3283926288,4.099783e+06,3283926288,...,36,37,116.025919,142.228961,164.640961,189.562271,189.562271,189.562271,547.252271,560.242271
9,2026-04-01,bhv1n,624,32.100694,0.051443,43622220000,6.990740e+07,2128614146,3.411241e+06,2128614146,...,40,40,607.773487,615.233487,615.233487,615.233487,598.478542,599.725305,612.715305,612.715305


In [ ]:
payer_df_all.keys()

Index(['submit_date', 'app_event_model_version', 'installs', 'sum_pred',
       'avg_pred', 'sum_tcpe', 'avg_tcpe', 'sum_cost', 'avg_cost',
       'sum_campaign_spend', 'sum_payer_label_d0', 'sum_payer_label_d1',
       'sum_payer_label_d2', 'sum_payer_label_d3', 'sum_payer_label_d4',
       'sum_payer_label_d5', 'sum_payer_label_d6', 'sum_payer_label_d7',
       'sum_capped_count_d0', 'sum_capped_count_d1', 'sum_capped_count_d2',
       'sum_capped_count_d3', 'sum_capped_count_d4', 'sum_capped_count_d5',
       'sum_capped_count_d6', 'sum_capped_count_d7', 'sum_capped_deposit_d0',
       'sum_capped_deposit_d1', 'sum_capped_deposit_d2',
       'sum_capped_deposit_d3', 'sum_capped_deposit_d4',
       'sum_capped_deposit_d5', 'sum_capped_deposit_d6',
       'sum_capped_deposit_d7'],
      dtype='object')

In [ ]:
payer_df_all[["app_event_model_version", "sum_pred", 'sum_payer_label_d0', 'sum_capped_count_d0', 'sum_payer_label_d1', 'sum_capped_count_d1', 'sum_payer_label_d2', 'sum_capped_count_d2',
       'sum_payer_label_d3', 'sum_capped_count_d3', 'sum_payer_label_d4', 'sum_capped_count_d4', 'sum_payer_label_d5', 'sum_capped_count_d5',
       'sum_payer_label_d6', 'sum_capped_count_d6', 'sum_payer_label_d7', 'sum_capped_count_d7']]

,app_event_model_version,sum_pred,sum_payer_label_d0,sum_capped_count_d0,sum_payer_label_d1,sum_capped_count_d1,sum_payer_label_d2,sum_capped_count_d2,sum_payer_label_d3,sum_capped_count_d3,sum_payer_label_d4,sum_capped_count_d4,sum_payer_label_d5,sum_capped_count_d5,sum_payer_label_d6,sum_capped_count_d6,sum_payer_label_d7,sum_capped_count_d7
0,bhv1n,40.027658,18,21,22,25,23,27,24,28,24,30,24,31,25,33,25,35
1,ctx1i,59.795531,33,35,37,39,37,39,37,39,37,39,37,39,38,48,38,52
2,v11-payer,89.721261,44,48,49,54,49,54,49,54,48,53,49,55,50,64,52,67
3,bhv1n,44.974848,12,12,15,20,17,22,17,22,17,22,17,22,17,26,17,26
4,ctx1i,48.297409,15,16,16,17,17,18,18,19,18,19,19,20,20,24,20,24
5,v11-payer,68.710391,34,35,35,36,36,37,37,38,39,40,40,44,41,53,43,56
6,bhv1n,38.430931,20,26,20,27,21,30,21,30,21,30,21,30,21,31,21,31
7,ctx1i,47.149240,15,16,15,16,15,16,15,16,15,16,15,16,17,21,17,22
8,v11-payer,58.506729,17,18,19,20,21,25,23,27,23,27,23,27,31,36,32,37
9,bhv1n,32.100694,12,24,13,30,13,30,13,30,13,30,14,32,15,40,15,40


In [ ]:
def draw_bias_comparison(dataframe: pd.DataFrame, outcome_column: str, prediction_column: str, title:str):
    new_list = []

    for payer_version in dataframe['app_event_model_version'].unique():
        temp_df = dataframe[dataframe['app_event_model_version'] == payer_version]
        for day in range(0, 8):
            current_outcome_col = outcome_column + str(day)
            # Get indices where the current outcome column is not 0 (and not NaN implicitly by pandas sum)
            # Also ensure that it's not NaN for robustness
            valid_indices = temp_df[(temp_df[current_outcome_col].notna()) & (temp_df[current_outcome_col] != 0)].index

            # Sum prediction_column and outcome_column only for valid_indices
            sum_prediction_for_day = temp_df.loc[valid_indices, prediction_column].sum()
            sum_outcome_for_day = temp_df.loc[valid_indices, current_outcome_col].sum()

            # print(day, sum_prediction_for_day, sum_outcome_for_day)

            if sum_outcome_for_day != 0: # This check is still good practice to catch any edge cases
                bias_value = ((sum_prediction_for_day / sum_outcome_for_day) - 1)
            else:
                bias_value = None # Or np.nan, depending on how you want to represent it

            new_list.append([f"d{day}", bias_value, payer_version])


    new_df = pd.DataFrame(new_list, columns=["dayX", "bias", "payer_version"])
    # print(new_df)

    fig = px.line(new_df, x='dayX', y='bias', color="payer_version", title=title, text=new_df['bias'].round(2))
    fig.update_traces(textposition='top center') # Position the text at the top center of each point
    display(fig)

In [ ]:
#v11_payer - Bias exploration Query

## Model Bias by dx

In [ ]:
draw_bias_comparison(
    payer_df_all,
    outcome_column="sum_payer_label_d",
    prediction_column="sum_pred",
    title="Payer Model Bias VS DayX - (Prediction (Payer rate) / Outcome(sum_payer_label_dx) - 1) "
)

## Product Bias by Dx

In [ ]:
def draw_product_bias_comparison_dx(dataframe: pd.DataFrame, title: str):
    new_list = []

    for payer_version in dataframe['app_event_model_version'].unique():
        temp_df = dataframe[dataframe['app_event_model_version'] == payer_version].copy()

        for day in range(0, 8):
            capped_count_col = f'sum_capped_count_d{day}'

            # Filter for rows where we have valid data for this specific day
            valid_data = temp_df[(temp_df[capped_count_col].notna()) & (temp_df[capped_count_col] != 0)]

            if not valid_data.empty:
                # Aggregated calculation: (Sum of Costs / Sum of Counts) / Avg Target CPE - 1
                # This matches the logic of 'Overall Product Bias'
                total_cost = valid_data['sum_cost'].sum()
                total_count = valid_data[capped_count_col].sum()
                avg_target_cpe = valid_data['avg_tcpe'].mean()

                observed_cpe = total_cost / total_count
                bias_value = (observed_cpe / avg_target_cpe) - 1
            else:
                bias_value = None

            new_list.append([f"d{day}", bias_value, payer_version])

    new_df = pd.DataFrame(new_list, columns=["dayX", "bias", "payer_version"])

    fig = px.line(new_df, x='dayX', y='bias', color="payer_version",
                  title=title, text=new_df['bias'].round(2))
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.update_traces(textposition='top center')
    display(fig)

draw_product_bias_comparison_dx(
    payer_df_all,
    title="Product Bias VS DayX - ((Observed CPE / Target CPE) - 1)"
)

# Prediction: Model payer rate prediction per day per model

In [ ]:
model_performance = payer_df_all.groupby('app_event_model_version').agg(
    avg_prediction=('avg_pred', 'mean'),
    avg_cpi=('avg_cost', 'mean'),
    avg_spend=('sum_cost', 'mean'),
    avg_tcpe=('avg_tcpe', 'mean'),
    avg_installs=('installs', 'mean')
).reset_index()

display(model_performance)

,app_event_model_version,avg_prediction,avg_cpi,avg_spend,avg_tcpe,avg_installs
0,bhv1n,0.043930,2.890335e+06,2201203832.533333,6.600556e+07,743.4
1,ctx1i,0.085557,3.972302e+06,1081045056.357143,7.002192e+07,318.357143
2,v11-payer,0.066062,4.997728e+06,2175119309.866667,7.257818e+07,464.4


## Average Prediction

In [ ]:
fig = px.line(
    payer_df_all,
    x='submit_date',
    y=f'avg_pred',
    color='app_event_model_version',
    title=f'Average Payer prediction VS Submit Date',
    labels={
        'submit_date': 'Submit Date',
        'app_event_model_version': 'App Event Model Version'
    },
    text=payer_df_all[f'avg_pred'].round(5) # Add absolute numbers as text, rounded to 2 decimal places
)
fig.update_traces(textposition='top center') # Position the text at the top center of each point
fig.show()

## Average target CPE

In [ ]:
fig = px.line(
    payer_df_all,
    x='submit_date',
    y=f'avg_tcpe',
    color='app_event_model_version',
    title=f'Average taget CPE VS Submit Date',
    labels={
        'submit_date': 'Submit Date',
        'app_event_model_version': 'App Event Model Version'
    },
    text=(payer_df_all[f'avg_tcpe'] / 1000000).round(2) # Add absolute numbers as text, rounded to 2 decimal places
)
fig.update_traces(textposition='top center') # Position the text at the top center of each point
fig.show()

## Average Spend

In [ ]:
print()
fig = px.line(
    payer_df_all,
    x='submit_date',
    y=f'avg_cost',
    color='app_event_model_version',
    title=f'Average Spend VS Submit Date',
    labels={
        'submit_date': 'Submit Date',
        'app_event_model_version': 'App Event Model Version'
    },
    text=(payer_df_all[f'avg_cost'] / 1000000).round(5) # Add absolute numbers as text, rounded to 2 decimal places
)
fig.update_traces(textposition='top center') # Position the text at the top center of each point
fig.show()

## Average Installs

In [ ]:
fig = px.line(
    payer_df_all,
    x='submit_date',
    y=f'installs',
    color='app_event_model_version',
    title=f'Average Installs VS Submit Date',
    labels={
        'submit_date': 'Submit Date',
        'app_event_model_version': 'App Event Model Version'
    },
    text=(payer_df_all[f'installs']).round(0) # Add absolute numbers as text, rounded to 2 decimal places
)
fig.update_traces(textposition='top center') # Position the text at the top center of each point
fig.show()

## Payer Rate

sum_payer_label_dx / installs

In [ ]:
def payer_rate_plot(payer_df_all):
  for i in range(0, 8):
    payer_df_all[f'payer_date_d{i}'] = ( payer_df_all[f'sum_payer_label_d{i}'] / payer_df_all[f'installs']) * 100

    # Calculate overall payer rate for each model for the current day i
    valid_data_df = payer_df_all[
        (payer_df_all[f'sum_payer_label_d{i}'].notna()) &
        (payer_df_all[f'sum_payer_label_d{i}'] != 0)
    ]
    overall_payer_rates = valid_data_df.groupby('app_event_model_version').agg(
        total_sum_payer_label = (f'sum_payer_label_d{i}', 'sum'),
        total_installs = ('installs', 'sum')
    ).reset_index()
    overall_payer_rates[f'overall_payer_rate_d{i}'] = (overall_payer_rates['total_sum_payer_label'] / overall_payer_rates['total_installs']) * 100

    # Create a subtitle string for the overall rates
    subtitle_parts = []
    for index, row in overall_payer_rates.iterrows():
        subtitle_parts.append(f'{row["app_event_model_version"]}: {row[f"overall_payer_rate_d{i}"]:.2f}%')
    subtitle_text = 'Average Rates: ' + '  |  '.join(subtitle_parts)

    fig = px.line(
        payer_df_all,
        x='submit_date',
        y=f'payer_date_d{i}',
        color='app_event_model_version',
        title=f'Payer Rate D{i} (sum_payer_label_d{i} / installs) * 100 VS Submit Date', # Assign only the main title here
        text=payer_df_all[f'payer_date_d{i}'].round(2) # Add bias numbers as text, rounded to 2 decimal places
    )

    # Add the subtitle as an annotation
    fig.update_layout(
        annotations=[
            dict(
                text=subtitle_text,
                xref='paper',
                yref='paper',
                x=0.5,
                y=0.96, # Adjust y to position it below the main title. This usually works well.
                showarrow=False,
                font=dict(size=15), # Smaller font size
                xanchor='center',
                yanchor='top' # Anchor the top of the text to the y position
            )
        ]
    )

    fig.add_hline(y=0)
    fig.update_traces(textposition='top center') # Position the text at the top center of each point
    fig.show()

payer_rate_plot(payer_df_all)

## Purchase Rate

sum_purchase_label_dx / installs

In [ ]:
def purchase_rate_plot(payer_df_all):
  for i in range(0, 8):
    payer_df_all[f'purchase_date_d{i}'] = ( payer_df_all[f'sum_capped_count_d{i}'] / payer_df_all[f'installs']) * 100

    # Calculate overall purchase rate for each model for the current day i
    valid_data_df = payer_df_all[
        (payer_df_all[f'sum_capped_count_d{i}'].notna()) &
        (payer_df_all[f'sum_capped_count_d{i}'] != 0)
    ]
    overall_purchase_rates = valid_data_df.groupby('app_event_model_version').agg(
        total_sum_purchase_label = (f'sum_capped_count_d{i}', 'sum'),
        total_installs = ('installs', 'sum')
    ).reset_index()
    overall_purchase_rates[f'overall_purchase_rate_d{i}'] = (overall_purchase_rates[f'total_sum_purchase_label'] / overall_purchase_rates['total_installs']) * 100

    # Create a subtitle string for the overall rates
    subtitle_parts = []
    for index, row in overall_purchase_rates.iterrows():
        subtitle_parts.append(f'{row["app_event_model_version"]}: {row[f"overall_purchase_rate_d{i}"]:.2f}%')
    subtitle_text = 'Average Rates: ' + '  |  '.join(subtitle_parts)

    fig = px.line(
        payer_df_all,
        x='submit_date',
        y=f'purchase_date_d{i}',
        color='app_event_model_version',
        title=f'Purchase Rate D{i} (sum_capped_count_d{i} / installs) * 100 VS Submit Date',
        text=payer_df_all[f'purchase_date_d{i}'].round(2) # Add bias numbers as text, rounded to 2 decimal places
    )

    # Add the subtitle as an annotation
    fig.update_layout(
        annotations=[
            dict(
                text=subtitle_text,
                xref='paper',
                yref='paper',
                x=0.5,
                y=0.96, # Adjust y to position it below the main title. This usually works well.
                showarrow=False,
                font=dict(size=15), # Smaller font size
                xanchor='center',
                yanchor='top' # Anchor the top of the text to the y position
            )
        ]
    )

    fig.add_hline(y=0)
    fig.update_traces(textposition='top center') # Position the text at the top center of each point
    fig.show()

purchase_rate_plot(payer_df_all)

In [ ]:
payer_df_all[["submit_date","app_event_model_version", "installs", "sum_capped_count_d7"]]

,submit_date,app_event_model_version,installs,sum_capped_count_d7
0,2026-03-29,bhv1n,869,35
1,2026-03-29,ctx1i,899,52
2,2026-03-29,v11-payer,1275,67
3,2026-03-30,bhv1n,967,26
4,2026-03-30,ctx1i,751,24
5,2026-03-30,v11-payer,976,56
6,2026-03-31,bhv1n,778,31
7,2026-03-31,ctx1i,685,22
8,2026-03-31,v11-payer,801,37
9,2026-04-01,bhv1n,624,40


In [ ]:
payer_df_all

,submit_date,app_event_model_version,installs,sum_pred,avg_pred,sum_tcpe,avg_tcpe,sum_cost,avg_cost,sum_campaign_spend,...,payer_date_d6,payer_date_d7,purchase_date_d0,purchase_date_d1,purchase_date_d2,purchase_date_d3,purchase_date_d4,purchase_date_d5,purchase_date_d6,purchase_date_d7
0,2026-03-29,bhv1n,869,40.027658,0.046062,53763940000,6.186875e+07,2258441514,2.598897e+06,2258441514,...,2.87687,2.87687,2.416571,2.87687,3.10702,3.222094,3.452244,3.567319,3.797468,4.027618
1,2026-03-29,ctx1i,899,59.795531,0.066513,43412650000,4.828993e+07,2503638045,2.784914e+06,2503638045,...,4.226919,4.226919,3.893215,4.338154,4.338154,4.338154,4.338154,4.338154,5.339266,5.784205
2,2026-03-29,v11-payer,1275,89.721261,0.070370,72520750000,5.687902e+07,5176174576,4.059745e+06,5176174576,...,3.921569,4.078431,3.764706,4.235294,4.235294,4.235294,4.156863,4.313725,5.019608,5.254902
3,2026-03-30,bhv1n,967,44.974848,0.046510,65889550000,6.813811e+07,2843675566,2.940719e+06,2843675566,...,1.758014,1.758014,1.240951,2.068252,2.275078,2.275078,2.275078,2.275078,2.688728,2.688728
4,2026-03-30,ctx1i,751,48.297409,0.064311,34872880000,4.643526e+07,1836228994,2.445045e+06,1836228994,...,2.663116,2.663116,2.130493,2.263648,2.396804,2.52996,2.52996,2.663116,3.195739,3.195739
5,2026-03-30,v11-payer,976,68.710391,0.070400,54316530000,5.565218e+07,3826569949,3.920666e+06,3826569949,...,4.20082,4.405738,3.586066,3.688525,3.790984,3.893443,4.098361,4.508197,5.430328,5.737705
6,2026-03-31,bhv1n,778,38.430931,0.049397,56160540000,7.218578e+07,2487350744,3.197109e+06,2487350744,...,2.699229,2.699229,3.341902,3.470437,3.856041,3.856041,3.856041,3.856041,3.984576,3.984576
7,2026-03-31,ctx1i,685,47.149240,0.068831,32594880000,4.758377e+07,1923600468,2.808176e+06,1923600468,...,2.481752,2.481752,2.335766,2.335766,2.335766,2.335766,2.335766,2.335766,3.065693,3.211679
8,2026-03-31,v11-payer,801,58.506729,0.073042,44542960000,5.560919e+07,3283926288,4.099783e+06,3283926288,...,3.870162,3.995006,2.247191,2.496879,3.121099,3.370787,3.370787,3.370787,4.494382,4.619226
9,2026-04-01,bhv1n,624,32.100694,0.051443,43622220000,6.990740e+07,2128614146,3.411241e+06,2128614146,...,2.403846,2.403846,3.846154,4.807692,4.807692,4.807692,4.807692,5.128205,6.410256,6.410256


# Outcome: Absolute Purchase event per day per model

In [ ]:
for i in range(0, 8):
    fig = px.line(
        payer_df_all,
        x='submit_date',
        y=f'sum_capped_count_d{i}',
        color='app_event_model_version',
        title=f'Absolute sum_capped_count_d{i} VS Submit Date',
        labels={
            'sum_capped_count_d' + str(i): f'sum_capped_count_d{i}',
            'submit_date': 'Submit Date',
            'app_event_model_version': 'App Event Model Version'
        },
        text=payer_df_all[f'sum_capped_count_d{i}'].round(2) # Add absolute numbers as text, rounded to 2 decimal places
    )
    fig.update_traces(textposition='top center') # Position the text at the top center of each point
    fig.show()

# Model bias

In [ ]:
def model_bias_plot(payer_df_all):
  for i in range(0, 8):
    payer_df_all[f'model_bias_d{i}'] = ( payer_df_all['sum_pred'] / payer_df_all[f'sum_payer_label_d{i}'] - 1) * 100

    # Calculate overall purchase rate for each model for the current day i
    valid_data_df = payer_df_all[
        (payer_df_all[f'sum_payer_label_d{i}'].notna()) &
        (payer_df_all[f'sum_payer_label_d{i}'] != 0)
    ]
    overall_model_bias = valid_data_df.groupby('app_event_model_version').agg(
        total_sum_payer_label = (f'sum_payer_label_d{i}', 'sum'),
        total_pred = ('sum_pred', 'sum')
    ).reset_index()
    overall_model_bias[f'overall_model_bias_d{i}'] = (overall_model_bias[f'total_pred'] / overall_model_bias['total_sum_payer_label'] - 1) * 100

    # Create a subtitle string for the overall rates
    subtitle_parts = []
    for index, row in overall_model_bias.iterrows():
        subtitle_parts.append(f'{row["app_event_model_version"]}: {row[f"overall_model_bias_d{i}"]:.2f}%')
    subtitle_text = 'Average Model Bias: ' + '  |  '.join(subtitle_parts)

    fig = px.line(
        payer_df_all,
        x='submit_date',
        y=f'model_bias_d{i}',
        color='app_event_model_version',
        title=f'Model Prediction Bias D{i} ((sum_pred / sum_payer_label_d{i} ）- 1) * 100 VS Submit Date',
        text=payer_df_all[f'model_bias_d{i}'].round(2) # Add bias numbers as text, rounded to 2 decimal places
    )

    # Add the subtitle as an annotation
    fig.update_layout(
        annotations=[
            dict(
                text=subtitle_text,
                xref='paper',
                yref='paper',
                x=0.5,
                y=0.96, # Adjust y to position it below the main title. This usually works well.
                showarrow=False,
                font=dict(size=15), # Smaller font size
                xanchor='center',
                yanchor='top' # Anchor the top of the text to the y position
            )
        ]
    )

    fig.add_hline(y=0)
    fig.update_traces(textposition='top center') # Position the text at the top center of each point
    fig.show()

model_bias_plot(payer_df_all)

In [ ]:
payer_df_all[["app_event_model_version", "sum_payer_label_d7", "sum_pred"]]

,app_event_model_version,sum_payer_label_d7,sum_pred
0,bhv1n,25,40.027658
1,ctx1i,38,59.795531
2,v11-payer,52,89.721261
3,bhv1n,17,44.974848
4,ctx1i,20,48.297409
5,v11-payer,43,68.710391
6,bhv1n,21,38.430931
7,ctx1i,17,47.149240
8,v11-payer,32,58.506729
9,bhv1n,15,32.100694


# Product Bias

In [ ]:
def product_bias_plot_per_day_v2(dataframe: pd.DataFrame):
    df = dataframe.copy()

    for day in range(0, 8):
        count_col = f'sum_capped_count_d{day}'
        bias_col = f'product_bias_d{day}'

        # Daily row-wise calculation: (Daily Cost / Daily Count) / Daily Avg Target CPE - 1
        df[bias_col] = (df['sum_cost'] / df[count_col]) / df['avg_tcpe'] - 1

        # Filter for display
        plot_df = df[df[bias_col].notna() & ~df[bias_col].isin([float('inf'), float('-inf')])]

        # Calculate overall aggregated bias across all dates for the subtitle
        valid_data_df = df[(df[count_col].notna()) & (df[count_col] != 0)]

        if not valid_data_df.empty:
            overall_stats = valid_data_df.groupby('app_event_model_version').agg(
                total_cost=('sum_cost', 'sum'),
                total_count=(count_col, 'sum'),
                mean_tcpe=('avg_tcpe', 'mean')
            ).reset_index()
            overall_stats['overall_bias'] = (overall_stats['total_cost'] / overall_stats['total_count']) / overall_stats['mean_tcpe'] - 1

            subtitle_parts = [f"{row['app_event_model_version']}: {row['overall_bias']:.2f}" for _, row in overall_stats.iterrows()]
            subtitle_text = 'Overall Aggregated Bias: ' + '  |  '.join(subtitle_parts)
        else:
            subtitle_text = "No valid data for overall bias"

        fig = px.line(
            plot_df,
            x='submit_date',
            y=bias_col,
            color='app_event_model_version',
            title=f'Product Bias D{day} Over Time: ((Observed CPE / Target CPE) - 1)',
            labels={'submit_date': 'Submit Date', bias_col: 'Bias'},
            text=plot_df[bias_col].round(2) if not plot_df.empty else None
        )

        fig.update_layout(
            annotations=[
                dict(
                    text=subtitle_text,
                    xref='paper', yref='paper',
                    x=0.5, y=0.96,
                    showarrow=False,
                    font=dict(size=13),
                    xanchor='center', yanchor='top'
                )
            ]
        )

        fig.add_hline(y=0, line_dash="dash", line_color="black")
        fig.update_traces(textposition='top center')

        # Ensure x-axis shows the relevant date range even if data is missing
        fig.update_xaxes(range=[df['submit_date'].min(), df['submit_date'].max()])

        fig.show()

product_bias_plot_per_day_v2(payer_df_all)

In [ ]:
payer_df_all.keys()

Index(['submit_date', 'app_event_model_version', 'installs', 'sum_pred',
       'avg_pred', 'sum_tcpe', 'avg_tcpe', 'sum_cost', 'avg_cost',
       'sum_campaign_spend', 'sum_payer_label_d0', 'sum_payer_label_d1',
       'sum_payer_label_d2', 'sum_payer_label_d3', 'sum_payer_label_d4',
       'sum_payer_label_d5', 'sum_payer_label_d6', 'sum_payer_label_d7',
       'sum_capped_count_d0', 'sum_capped_count_d1', 'sum_capped_count_d2',
       'sum_capped_count_d3', 'sum_capped_count_d4', 'sum_capped_count_d5',
       'sum_capped_count_d6', 'sum_capped_count_d7', 'sum_capped_deposit_d0',
       'sum_capped_deposit_d1', 'sum_capped_deposit_d2',
       'sum_capped_deposit_d3', 'sum_capped_deposit_d4',
       'sum_capped_deposit_d5', 'sum_capped_deposit_d6',
       'sum_capped_deposit_d7', 'payer_date_d0', 'payer_date_d1',
       'payer_date_d2', 'payer_date_d3', 'payer_date_d4', 'payer_date_d5',
       'payer_date_d6', 'payer_date_d7', 'purchase_date_d0',
       'purchase_date_d1', 'purchase_d

### Product Bias Analysis per Day (Time-Series with Overall Aggregation)

In [ ]:
def product_bias_plot_per_day_v2(dataframe: pd.DataFrame):
    df = dataframe.copy()

    for day in range(0, 8):
        count_col = f'sum_capped_count_d{day}'
        bias_col = f'product_bias_d{day}'

        # Daily row-wise calculation: (Daily Cost / Daily Count) / Daily Avg Target CPE - 1
        df[bias_col] = (df['sum_cost'] / df[count_col]) / df['avg_tcpe'] - 1

        # Filter for display
        plot_df = df[df[bias_col].notna() & ~df[bias_col].isin([float('inf'), float('-inf')])]

        # Calculate overall aggregated bias across all dates for the subtitle
        valid_data_df = df[(df[count_col].notna()) & (df[count_col] != 0)]

        if not valid_data_df.empty:
            overall_stats = valid_data_df.groupby('app_event_model_version').agg(
                total_cost=('sum_cost', 'sum'),
                total_count=(count_col, 'sum'),
                mean_tcpe=('avg_tcpe', 'mean')
            ).reset_index()
            overall_stats['overall_bias'] = (overall_stats['total_cost'] / overall_stats['total_count']) / overall_stats['mean_tcpe'] - 1

            subtitle_parts = [f"{row['app_event_model_version']}: {row['overall_bias']:.2f}" for _, row in overall_stats.iterrows()]
            subtitle_text = 'Overall Aggregated Bias: ' + '  |  '.join(subtitle_parts)
        else:
            subtitle_text = "No valid data for overall bias"

        fig = px.line(
            plot_df,
            x='submit_date',
            y=bias_col,
            color='app_event_model_version',
            title=f'Product Bias D{day} Over Time: ((Observed CPE / Target CPE) - 1)',
            labels={'submit_date': 'Submit Date', bias_col: 'Bias'},
            text=plot_df[bias_col].round(2) if not plot_df.empty else None
        )

        fig.update_layout(
            annotations=[
                dict(
                    text=subtitle_text,
                    xref='paper', yref='paper',
                    x=0.5, y=0.96,
                    showarrow=False,
                    font=dict(size=13),
                    xanchor='center', yanchor='top'
                )
            ]
        )

        fig.add_hline(y=0, line_dash="dash", line_color="black")
        fig.update_traces(textposition='top center')

        # Ensure x-axis shows the relevant date range even if data is missing
        fig.update_xaxes(range=[df['submit_date'].min(), df['submit_date'].max()])

        fig.show()

product_bias_plot_per_day_v2(payer_df_all)